# 02: Preprocessing


In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import preprocessing

# The raw recordings live one level up from notebooks/.
RAW = "../data/raw"

## Load one run

`load_run` is the single entry point the notebooks call. It returns a `Run`
dataclass holding the cleaned IMU streams, the cleaned BLE stream, a shared time
origin, and metadata.

In [2]:
run = preprocessing.load_run(1, raw_dir=RAW)
print("run_id     :", run.run_id)
print("t0_ms      :", run.t0_ms)
print("duration_s :", round(run.duration_s, 2))

run_id     : 1
t0_ms      : 1782119629473
duration_s : 197.76


## IMU streams

In [3]:
run.accel.head()

,t_ms,t_rel,x,y,z
0,1782119629473,0.000,-3.708319,3.494936,8.582913
1,1782119629473,0.000,-3.772963,3.109469,9.445425
2,1782119629475,0.002,-3.986047,3.095104,9.903017
3,1782119629475,0.002,-3.919009,3.221997,9.895834
4,1782119629485,0.012,-3.665224,3.396774,9.296985


In [4]:
run.gyro.head()

,t_ms,t_rel,x,y,z
0,1782119629475,0.002,-0.495400,-0.744065,-0.098201
1,1782119629475,0.002,-0.281288,-0.622629,0.005126
2,1782119629485,0.012,-0.123567,-0.532084,0.077030
3,1782119629496,0.023,-0.048468,-0.518768,0.095671
4,1782119629508,0.035,-0.071903,-0.575226,0.058388


## BLE stream

In [5]:
run.ble.head()

,t_ms,t_rel,beacon,address,rssi
0,1782119629515,0.042,arrive_emi1,F3:6E:C9:45:67:45,-86
1,1782119629535,0.062,arrive_emi8,E9:C4:63:D1:6D:A2,-76
2,1782119629599,0.126,arrive_emi1,F3:6E:C9:45:67:45,-85
3,1782119629717,0.244,arrive_emi1,F3:6E:C9:45:67:45,-84
4,1782119629739,0.266,arrive_emi8,E9:C4:63:D1:6D:A2,-76


## Run metadata

`build_metadata` records descriptive facts only (it never changes the data):
sample counts per stream, how many BLE rows were dropped, which beacons were seen,
the RSSI range, the largest BLE gap, and the duplicate flag.

In [6]:
import pprint
pprint.pprint(run.meta)

{'beacon_obs_counts': {'arrive_emi1': 359,
                       'arrive_emi10': 591,
                       'arrive_emi2': 381,
                       'arrive_emi3': 376,
                       'arrive_emi4': 579,
                       'arrive_emi8': 297},
 'beacons_seen': ['arrive_emi1',
                  'arrive_emi10',
                  'arrive_emi2',
                  'arrive_emi3',
                  'arrive_emi4',
                  'arrive_emi8'],
 'ble_clean_rows': 2583,
 'ble_dropped_rows': 81,
 'duplicate_with': [],
 'imu_counts': {'accel': 19868,
                'gyro': 19866,
                'imu_processed': 59502,
                'mag': 19776},
 'is_duplicate': False,
 'max_ble_gap_s': 3.844,
 'n_raw_rows': 127472,
 'rssi_max': -55,
 'rssi_min': -102,
 'source_counts': {'beacon': 5789,
                   'ble_rssi': 2664,
                   'imu': 119012,
                   'live_ble_snapshot': 7}}


## All four runs at a glance

Loading every run and summarising the key facts. Durations line up with the
reference spreadsheet, and only 6 of the 8 installed beacons are ever seen (the
east-wing pair is not covered by these paths).

In [7]:
rows = []
for run_id in [1, 2, 3, 4]:
    r = preprocessing.load_run(run_id, raw_dir=RAW)
    rows.append({
        "run": run_id,
        "duration_s": round(r.duration_s, 1),
        "accel": r.meta["imu_counts"]["accel"],
        "gyro": r.meta["imu_counts"]["gyro"],
        "ble_clean": r.meta["ble_clean_rows"],
        "ble_dropped": r.meta["ble_dropped_rows"],
        "beacons_seen": len(r.meta["beacons_seen"]),
        "max_ble_gap_s": r.meta["max_ble_gap_s"],
    })
pd.DataFrame(rows)

,run,duration_s,accel,gyro,ble_clean,ble_dropped,beacons_seen,max_ble_gap_s
0,1,197.8,19868,19866,2583,81,6,3.844
1,2,190.5,19136,19126,2008,76,6,0.968
2,3,252.6,25377,25366,2381,106,6,4.202
3,4,227.9,22890,22882,2220,125,6,1.015


## Time synchronisation

All streams share one time origin `t0` (the earliest IMU/BLE timestamp of the run)
and a relative clock `t_rel = (timestamp_ms - t0) / 1000`. The IMU is the high-rate
base clock; BLE is event-driven and kept at its own irregular times — we do **not**
resample BLE onto a fixed grid. This event-driven layout is exactly what the
particle filter needs later: predict on each step, and update only when a BLE
reading actually arrives.

## Data-quality notes

- **Run 2 duplicate (resolved):** the original `Record_data_path_2.csv` was
  byte-identical to Run 1 and was replaced with the correct recording, so
  `is_duplicate` is now `False` for every run.
- **Run 2 start offset (resolved):** its clock aligns with the reference, so
  `start_offset_s = 0` (see `docs/experiment_protocol.md`).
- **Non-project BLE:** unrelated scanned devices are filtered out; only
  `arrive_emi*` beacons are kept.

Next: **03 — Step detection and the motion model.**